In [1]:
import pandas as pd
from sklearn import svm
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV
from sklearn.decomposition import PCA


In [3]:
# Ladataan data
train_df = pd.read_csv('data/train_data.csv')
val_df = pd.read_csv('data/val_data.csv')
test_df = pd.read_csv('data/test_data.csv')

# Jaetaan data ominaisuuksiin ja luokkiin, putetaan "Attack Type" pois ominaisuuksista
X_train = train_df.drop('Attack Type', axis=1)
X_val = val_df.drop('Attack Type', axis=1)
X_test = test_df.drop('Attack Type', axis=1)

# Jaetaan luokat
y_train = train_df['Attack Type']
y_val = val_df['Attack Type']
y_test = test_df['Attack Type']




In [ ]:
# Pipeline, sisältää skaalauksen, pca:n ja svm:n
pipeline = Pipeline([
    ('scaler', StandardScaler(),),
    ('pca', PCA(n_components=0.95)),
    ('svm', SVC(class_weight='balanced', random_state=42)) 

])

# Määritellään hyperparametrit, joita haluamme kokeilla GridSearchCV:ssä
param_grid = {
    'scaler': [StandardScaler(), MinMaxScaler()], # Kokeillaan kahta eri skaalainmenetelmää
    'svm__C': [0.1, 1, 5, 10], # C-arvojen kokeilu
    'svm__gamma': ['scale', 'auto'], # Gamma-arvojen kokeilu, scale ja auto ovat yleisiä valintoja SVM:ssä
    'svm__kernel': ['rbf'] # Käytetään RBF-kernelia, joka on yleinen SVM:ssä, linear toistaiseksi jätetty pois aikaa säästääksemme
}

# Suoritetaan GridSearchCV hyperparametrien optimointiin
grid = GridSearchCV(pipeline, param_grid, cv=5, verbose=2) # cv=5 tarkoittaa 5-kertaista ristiinvalidointia, verbose=2 antaa enemmän tietoa prosessin etenemisestä
grid.fit(X_train, y_train)

# Arvioidaan mallin suorituskykyä validointidatalla
y_pred = grid.predict(X_val) # Ennustetaan luokat validointidatalla
print(classification_report(y_val, y_pred))

print("Best parameters:", grid.best_params_)
print("Best accuracy on training data:", grid.best_score_)

Fitting 5 folds for each of 16 candidates, totalling 80 fits
[CV] END scaler=StandardScaler(), svm__C=0.1, svm__gamma=scale, svm__kernel=rbf; total time= 1.0min
[CV] END scaler=StandardScaler(), svm__C=0.1, svm__gamma=scale, svm__kernel=rbf; total time=  59.7s
[CV] END scaler=StandardScaler(), svm__C=0.1, svm__gamma=scale, svm__kernel=rbf; total time=  58.6s
[CV] END scaler=StandardScaler(), svm__C=0.1, svm__gamma=scale, svm__kernel=rbf; total time=  59.7s
[CV] END scaler=StandardScaler(), svm__C=0.1, svm__gamma=scale, svm__kernel=rbf; total time=  57.7s
[CV] END scaler=StandardScaler(), svm__C=0.1, svm__gamma=auto, svm__kernel=rbf; total time=  58.2s
[CV] END scaler=StandardScaler(), svm__C=0.1, svm__gamma=auto, svm__kernel=rbf; total time=  57.9s
[CV] END scaler=StandardScaler(), svm__C=0.1, svm__gamma=auto, svm__kernel=rbf; total time=  56.6s
[CV] END scaler=StandardScaler(), svm__C=0.1, svm__gamma=auto, svm__kernel=rbf; total time=  57.2s
[CV] END scaler=StandardScaler(), svm__C=0.

In [4]:
y_test_pred = grid.predict(X_test)

print("\n--- Final test results (Test Data) ---")
print(classification_report(y_test, y_test_pred))


--- Final test results (Test Data) ---
                precision    recall  f1-score   support

          Bots       0.03      0.39      0.05        18
   Brute Force       0.65      0.77      0.71        75
          DDoS       0.65      0.88      0.75      1023
           DoS       0.82      0.87      0.84      1535
Normal Traffic       0.98      0.86      0.92     16616
 Port Scanning       0.37      1.00      0.54       715
   Web Attacks       0.07      0.94      0.14        18

      accuracy                           0.86     20000
     macro avg       0.51      0.82      0.56     20000
  weighted avg       0.93      0.86      0.89     20000



In [10]:
# Tallennetaan tulokset tekstitiedostoon
with open('svm_projektin_tulokset_2.txt', 'w') as f:
    f.write("--- SVM MALLIN TULOKSET ---\n\n")
    f.write(f"Parhaat parametrit: {grid.best_params_}\n")
    f.write(f"Paras tarkkuus treenidatalla: {grid.best_score_:.4f}\n\n")
    f.write("--- LOPULLINEN KOE (Test Data) ---\n")
    f.write(classification_report(y_test, y_test_pred))

print("Tulokset tallennettu tiedostoon: svm_projektin_tulokset_2.txt")

Tulokset tallennettu tiedostoon: svm_projektin_tulokset_2.txt


In [ ]:
import joblib

# Tallennetaan valmis malli tiedostoon
joblib.dump(grid, 'svm_optimoitu_malli_2.pkl')

print("Malli tallennettu tiedostoon: svm_optimoitu_malli_2.pkl")

# Myöhemmin voit ladata sen näin:
# ladattu_malli = joblib.load('svm_optimoitu_malli_2.pkl')

Malli tallennettu tiedostoon: svm_optimoitu_malli_2.pkl


Testataan vielä SMOTEN kanssa

In [4]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
import joblib

pipeline_imb = ImbPipeline([
    ('smote', SMOTE(random_state=42)), # SMOTE tasapainottaa
    ('scaler', MinMaxScaler()), # Skaalataan data
    ('pca', PCA(n_components=0.95)), # PCA:lla säilytetään 95% varianssista
    ('svm', SVC(class_weight='balanced', random_state=42)) # SVM-luokittelu
])

param_grid_smote = {
    'svm__C': [0.1, 1, 5, 10], # C-arvojen kokeilu
    'svm__gamma': ['scale', 'auto'], # Gamma-arvojen kokeilu
    'svm__kernel': ['rbf'] # Käytetään RBF-kernel
}

print("\n--- SMOTE + SVM OPTIMOINTI (GridSearchCV) ---\n")
grid_smote = GridSearchCV(pipeline_imb, param_grid_smote, cv=5, verbose=2)
grid_smote.fit(X_train, y_train)

print("\n--- SMOTE + SVM OPTIMOINTI ---\n")
y_pred_smote = pipeline_imb.fit(X_train, y_train).predict(X_val)
print(classification_report(y_val, y_pred_smote))

print("Best parameters with SMOTE:", grid_smote.best_params_())


--- SMOTE + SVM OPTIMOINTI (GridSearchCV) ---

Fitting 5 folds for each of 8 candidates, totalling 40 fits
[CV] END ......svm__C=0.1, svm__gamma=scale, svm__kernel=rbf; total time=12.9min
[CV] END ......svm__C=0.1, svm__gamma=scale, svm__kernel=rbf; total time=12.4min
[CV] END ......svm__C=0.1, svm__gamma=scale, svm__kernel=rbf; total time=10.3min
[CV] END ......svm__C=0.1, svm__gamma=scale, svm__kernel=rbf; total time=13.0min
[CV] END ......svm__C=0.1, svm__gamma=scale, svm__kernel=rbf; total time=11.4min
[CV] END .......svm__C=0.1, svm__gamma=auto, svm__kernel=rbf; total time=19.3min
[CV] END .......svm__C=0.1, svm__gamma=auto, svm__kernel=rbf; total time=18.4min
[CV] END .......svm__C=0.1, svm__gamma=auto, svm__kernel=rbf; total time=17.9min
[CV] END .......svm__C=0.1, svm__gamma=auto, svm__kernel=rbf; total time=20.6min
[CV] END .......svm__C=0.1, svm__gamma=auto, svm__kernel=rbf; total time=18.4min
[CV] END ........svm__C=1, svm__gamma=scale, svm__kernel=rbf; total time=11.7min
[

TypeError: 'dict' object is not callable

In [ ]:
# korjattu versio:
y_pred_smote_oikea = grid_smote.predict(X_val)

# tulostetaan korjattu raportti
print("\n--- OIKEAT SMOTE VALIDOINTITULOKSET (GridSearchCV) ---")
print(classification_report(y_val, y_pred_smote_oikea))

# tulostetaan parhaat parametrit
print("Best parameters with SMOTE:", grid_smote.best_params_)

# tallennetaan malli
import joblib
joblib.dump(grid_smote, 'svm_optimoitu_malli_SMOTE_oikea.pkl')
print("\nMalli tallennettu onnistuneesti!")


--- OIKEAT SMOTE VALIDOINTITULOKSET (GridSearchCV) ---
                precision    recall  f1-score   support

          Bots       0.03      0.71      0.05        17
   Brute Force       0.35      0.51      0.41        76
          DDoS       0.63      0.81      0.70      1023
           DoS       0.77      0.86      0.81      1534
Normal Traffic       0.96      0.89      0.92     16616
 Port Scanning       0.49      0.53      0.51       715
   Web Attacks       0.06      0.89      0.11        19

      accuracy                           0.87     20000
     macro avg       0.47      0.74      0.50     20000
  weighted avg       0.91      0.87      0.88     20000

Best parameters with SMOTE: {'svm__C': 0.1, 'svm__gamma': 'scale', 'svm__kernel': 'rbf'}

Malli tallennettu onnistuneesti!


In [6]:
import joblib
from sklearn.metrics import classification_report

# 1. Haetaan lopulliset ennusteet (Test Data)
y_test_pred_smote = grid_smote.predict(X_test)

# 2. Tallennetaan kaikki tekstitiedostoon
with open('svm_SMOTE_tulokset_lopullinen.txt', 'w') as f:
    f.write("--- SVM + SMOTE LOPULLISET TULOKSET ---\n\n")
    f.write(f"Parhaat parametrit: {grid_smote.best_params_}\n")
    f.write(f"Paras tarkkuus treenidatalla (CV score): {grid_smote.best_score_:.4f}\n\n")
    f.write("--- LOPULLINEN KOE (Näkymätön Test Data) ---\n")
    f.write(classification_report(y_test, y_test_pred_smote))

print("Tekstitulokset tallennettu: svm_SMOTE_tulokset_lopullinen.txt")

Tekstitulokset tallennettu: svm_SMOTE_tulokset_lopullinen.txt
